In [ ]:
# -*- coding: utf-8 -*-
"""
Análise de Dados do Titanic com Pandas e NumPy
Dataset: Base de Dados Titanic.csv
Sem uso de matplotlib ou seaborn
"""

import pandas as pd
import numpy as np

# Configurações de visualização
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.2f}'.format)

# ============================================================================
# 1. CARREGAMENTO DOS DADOS
# ============================================================================

df = pd.read_csv('Base de Dados Titanic.csv')

print("=" * 80)
print("ANÁLISE EXPLORATÓRIA E TRATAMENTO DE DADOS DO TITANIC")
print("=" * 80)

# ============================================================================
# 2. DESCRIÇÃO DOS DADOS
# ============================================================================

print("\n" + "=" * 80)
print("2. DESCRIÇÃO DOS DADOS")
print("=" * 80)

print("\n2.1. Primeiras linhas do dataset:")
print(df.head(10).to_string())

print("\n2.2. Últimas linhas do dataset:")
print(df.tail(10).to_string())

print("\n2.3. Dimensões do dataset:")
print(f"Linhas: {df.shape[0]}, Colunas: {df.shape[1]}")

print("\n2.4. Colunas do dataset:")
print(df.columns.tolist())

print("\n2.5. Tipos de variáveis (dtypes):")
print(df.dtypes)

print("\n2.6. Quantidade de valores distintos por coluna:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()} valores distintos")

print("\n2.7. Valores nulos por coluna:")
null_counts = df.isnull().sum()
null_percent = (df.isnull().sum() / len(df)) * 100
null_df = pd.DataFrame({
    'Quantidade de Nulos': null_counts,
    'Percentual (%)': null_percent.round(2)
})
print(null_df[null_df['Quantidade de Nulos'] > 0])

print("\n2.8. Estatísticas descritivas das variáveis numéricas:")
print(df.describe())

print("\n2.9. Estatísticas das variáveis categóricas:")
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f"\n  {col}:")
    print(f"    Valores únicos: {df[col].nunique()}")
    print(f"    Valores mais frequentes:")
    print(df[col].value_counts().head())

# ============================================================================
# 3. TRATAMENTO DE QUALIDADE DOS DADOS
# ============================================================================

print("\n" + "=" * 80)
print("3. TRATAMENTO DE QUALIDADE DOS DADOS")
print("=" * 80)

# Cópia do dataframe original para tratamento
df_clean = df.copy()

print("\n3.1. Verificação de valores constantes (variância zero):")
constante_encontrada = False
for col in df_clean.columns:
    if df_clean[col].nunique() <= 1:
        print(f"  ✓ Coluna '{col}' é CONSTANTE - será removida")
        df_clean = df_clean.drop(columns=[col])
        constante_encontrada = True
        
if not constante_encontrada:
    print("  Nenhuma coluna constante encontrada")

print("\n3.2. Tratamento de valores nulos:")
print("  Situação antes do tratamento:")
print(f"  Total de valores nulos: {df_clean.isnull().sum().sum()}")

# Tratamento da coluna Age (idade)
age_median = df_clean['Age'].median()
null_age_before = df_clean['Age'].isnull().sum()
df_clean['Age'] = df_clean['Age'].fillna(age_median)
print(f"  ✓ Coluna 'Age': Preenchidos {null_age_before} valores nulos com a mediana ({age_median:.1f})")

# Tratamento da coluna Cabin (cabine)
null_cabin_before = df_clean['Cabin'].isnull().sum()
df_clean['Cabin'] = df_clean['Cabin'].fillna('Desconhecida')
print(f"  ✓ Coluna 'Cabin': Preenchidos {null_cabin_before} valores nulos com 'Desconhecida'")

# Tratamento da coluna Embarked (porto de embarque)
null_embarked_before = df_clean['Embarked'].isnull().sum()
embarked_mode = df_clean['Embarked'].mode()[0]
df_clean['Embarked'] = df_clean['Embarked'].fillna(embarked_mode)
print(f"  ✓ Coluna 'Embarked': Preenchidos {null_embarked_before} valores nulos com a moda ({embarked_mode})")

# Tratamento da coluna Fare (tarifa) - apenas se houver nulos
if df_clean['Fare'].isnull().sum() > 0:
    fare_median = df_clean['Fare'].median()
    null_fare_before = df_clean['Fare'].isnull().sum()
    df_clean['Fare'] = df_clean['Fare'].fillna(fare_median)
    print(f"  ✓ Coluna 'Fare': Preenchidos {null_fare_before} valores nulos com a mediana ({fare_median:.2f})")

print("\n  Situação após o tratamento:")
print(f"  Total de valores nulos: {df_clean.isnull().sum().sum()}")

print("\n3.3. Criação de novas colunas para análise:")

# Criação de faixas etárias
bins = [0, 12, 18, 35, 50, 100]
labels = ['Criança', 'Adolescente', 'Jovem Adulto', 'Adulto', 'Idoso']
df_clean['Faixa_Etaria'] = pd.cut(df_clean['Age'], bins=bins, labels=labels, right=False)

# Criação de categoria de tarifa
df_clean['Categoria_Tarifa'] = pd.qcut(df_clean['Fare'], q=4, labels=['Econômica', 'Média', 'Alta', 'Premium'])

# Tamanho da família
df_clean['Tamanho_Familia'] = df_clean['SibSp'] + df_clean['Parch'] + 1

# Categoria do tamanho da família
def categoriza_familia(tamanho):
    if tamanho == 1:
        return 'Sozinho'
    elif tamanho <= 3:
        return 'Família Pequena'
    else:
        return 'Família Grande'

df_clean['Categoria_Familia'] = df_clean['Tamanho_Familia'].apply(categoriza_familia)

# Título extraído do nome
df_clean['Titulo'] = df_clean['Name'].str.extract(r',\s*([^\.]+)\.', expand=False)

print("  ✓ Criadas colunas: Faixa_Etaria, Categoria_Tarifa, Tamanho_Familia, Categoria_Familia, Titulo")

print("\n3.4. Dataset após limpeza:")
print(f"  Formato: {df_clean.shape}")
print(df_clean.head().to_string())

# ============================================================================
# 4. PERGUNTAS SOBRE O DATASET
# ============================================================================

print("\n" + "=" * 80)
print("4. PERGUNTAS E RESPOSTAS")
print("=" * 80)

# ============================================================
# PERGUNTA 1: Qual a taxa de sobrevivência por classe social?
# ============================================================
print("\n" + "-" * 80)
print("PERGUNTA 1: Qual a taxa de sobrevivência por classe social?")
print("-" * 80)

taxa_sobrevivencia_classe = df_clean.groupby('Pclass')['Survived'].agg(['mean', 'count', 'sum'])
taxa_sobrevivencia_classe.columns = ['Taxa_Sobrevivencia', 'Total_Passageiros', 'Sobreviventes']
taxa_sobrevivencia_classe['Taxa_Sobrevivencia'] = (taxa_sobrevivencia_classe['Taxa_Sobrevivencia'] * 100).round(2)
taxa_sobrevivencia_classe['Nao_Sobreviventes'] = taxa_sobrevivencia_classe['Total_Passageiros'] - taxa_sobrevivencia_classe['Sobreviventes']
print(taxa_sobrevivencia_classe)

print("\n📊 Resposta:")
for classe in [1, 2, 3]:
    print("  - {}ª Classe: Taxa de sobrevivência de {:.2f}%".format(
        classe, taxa_sobrevivencia_classe.loc[classe, 'Taxa_Sobrevivencia']))
print("  Conclusão: Passageiros da 1ª classe tiveram maior chance de sobrevivência,")
print("  enquanto os da 3ª classe tiveram a menor taxa, sugerindo que a classe")
print("  social influenciou significativamente no resgate.")

# Visualização em texto
print("\n  Gráfico de Barras (Texto):")
max_rate = taxa_sobrevivencia_classe['Taxa_Sobrevivencia'].max()
for classe in [1, 2, 3]:
    rate = taxa_sobrevivencia_classe.loc[classe, 'Taxa_Sobrevivencia']
    bar_length = int((rate / max_rate) * 50)
    bar = '█' * bar_length
    print(f"  {classe}ª Classe | {bar} {rate:.1f}%")

# ============================================================
# PERGUNTA 2: Homens ou mulheres tiveram maior chance de sobreviver?
# ============================================================
print("\n" + "-" * 80)
print("PERGUNTA 2: Homens ou mulheres tiveram maior chance de sobreviver?")
print("-" * 80)

taxa_sobrevivencia_sexo = df_clean.groupby('Sex')['Survived'].agg(['mean', 'count', 'sum'])
taxa_sobrevivencia_sexo.columns = ['Taxa_Sobrevivencia', 'Total', 'Sobreviventes']
taxa_sobrevivencia_sexo['Taxa_Sobrevivencia'] = (taxa_sobrevivencia_sexo['Taxa_Sobrevivencia'] * 100).round(2)

# Análise combinada Sexo e Classe
taxa_sexo_classe = df_clean.pivot_table(values='Survived', 
                                         index='Pclass', 
                                         columns='Sex', 
                                         aggfunc='mean') * 100

print("\nTaxa de sobrevivência por sexo:")
print(taxa_sobrevivencia_sexo)
print("\nTaxa de sobrevivência por sexo e classe (%):")
print(taxa_sexo_classe.round(2))

print("\n📊 Resposta:")
print(f"  - Mulheres: {taxa_sobrevivencia_sexo.loc['female', 'Taxa_Sobrevivencia']:.2f}% de sobrevivência")
print(f"  - Homens: {taxa_sobrevivencia_sexo.loc['male', 'Taxa_Sobrevivencia']:.2f}% de sobrevivência")
print("  Conclusão: As mulheres tiveram uma taxa de sobrevivência drasticamente")
print("  superior à dos homens, confirmando o protocolo 'mulheres e crianças primeiro'.")

# Visualização em texto
print("\n  Gráfico de Barras (Texto):")
for sex in ['female', 'male']:
    rate = taxa_sobrevivencia_sexo.loc[sex, 'Taxa_Sobrevivencia']
    bar_length = int((rate / 100) * 50)
    bar = '█' * bar_length
    sex_label = 'Mulheres' if sex == 'female' else 'Homens'
    print(f"  {sex_label:10} | {bar} {rate:.1f}%")

# ============================================================
# PERGUNTA 3: Qual o perfil dos passageiros que mais sobreviveram?
# ============================================================
print("\n" + "-" * 80)
print("PERGUNTA 3: Qual o perfil dos passageiros que mais sobreviveram?")
print("-" * 80)

# Análise por faixa etária
taxa_faixa_etaria = df_clean.groupby('Faixa_Etaria', observed=False)['Survived'].agg(['mean', 'count'])
taxa_faixa_etaria['mean'] = (taxa_faixa_etaria['mean'] * 100).round(2)

# Análise por tamanho da família
taxa_familia = df_clean.groupby('Categoria_Familia')['Survived'].agg(['mean', 'count'])
taxa_familia['mean'] = (taxa_familia['mean'] * 100).round(2)

# Análise por porto de embarque
taxa_porto = df_clean.groupby('Embarked')['Survived'].agg(['mean', 'count'])
taxa_porto['mean'] = (taxa_porto['mean'] * 100).round(2)

# Categoria de tarifa
taxa_tarifa = df_clean.groupby('Categoria_Tarifa', observed=False)['Survived'].agg(['mean', 'count'])
taxa_tarifa['mean'] = (taxa_tarifa['mean'] * 100).round(2)

# Análise por título
taxa_titulo = df_clean.groupby('Titulo')['Survived'].agg(['mean', 'count'])
taxa_titulo['mean'] = (taxa_titulo['mean'] * 100).round(2)
# Filtrar apenas títulos com mais de 5 ocorrências
taxa_titulo_filtrado = taxa_titulo[taxa_titulo['count'] > 5].sort_values('mean', ascending=False)

print("\nTaxa de sobrevivência por Faixa Etária:")
print(taxa_faixa_etaria)
print("\nTaxa de sobrevivência por Tamanho da Família:")
print(taxa_familia)
print("\nTaxa de sobrevivência por Porto de Embarque:")
print(taxa_porto)
print("\nTaxa de sobrevivência por Categoria de Tarifa:")
print(taxa_tarifa)
print("\nTaxa de sobrevivência por Título (mais de 5 ocorrências):")
print(taxa_titulo_filtrado)

print("\n📊 Resposta:")
print("  Perfil com maior chance de sobrevivência:")
print(f"  - Sexo: Feminino ({taxa_sobrevivencia_sexo.loc['female', 'Taxa_Sobrevivencia']:.1f}%)")
print(f"  - Classe: 1ª Classe ({taxa_sobrevivencia_classe.loc[1, 'Taxa_Sobrevivencia']:.1f}%)")
print(f"  - Faixa Etária: {taxa_faixa_etaria['mean'].idxmax()} ({taxa_faixa_etaria['mean'].max():.1f}%)")
print(f"  - Família: {taxa_familia['mean'].idxmax()} ({taxa_familia['mean'].max():.1f}%)")
print(f"  - Tarifa: {taxa_tarifa['mean'].idxmax()} ({taxa_tarifa['mean'].max():.1f}%)")
print(f"  - Porto de Embarque: {taxa_porto['mean'].idxmax()} ({taxa_porto['mean'].max():.1f}%)")

# Visualização consolidada em texto
print("\n  Resumo Visual (Texto):")
print(f"  {'Perfil':<25} {'Taxa':>10}")
print(f"  {'-'*35}")
for label, rate in [
    ('Sexo: Feminino', taxa_sobrevivencia_sexo.loc['female', 'Taxa_Sobrevivencia']),
    ('Classe: 1ª', taxa_sobrevivencia_classe.loc[1, 'Taxa_Sobrevivencia']),
    ('Idade: ' + str(taxa_faixa_etaria['mean'].idxmax()), taxa_faixa_etaria['mean'].max()),
    ('Família: ' + str(taxa_familia['mean'].idxmax()), taxa_familia['mean'].max()),
    ('Tarifa: ' + str(taxa_tarifa['mean'].idxmax()), taxa_tarifa['mean'].max()),
    ('Embarque: ' + str(taxa_porto['mean'].idxmax()), taxa_porto['mean'].max()),
]:
    bar_length = int((rate / 100) * 30)
    bar = '█' * bar_length
    print(f"  {label:<25} {bar} {rate:.1f}%")

# ============================================================
# PERGUNTA 4: Existe relação entre o valor da tarifa e a sobrevivência?
# ============================================================
print("\n" + "-" * 80)
print("PERGUNTA 4: Existe relação entre o valor da tarifa e a sobrevivência?")
print("-" * 80)

# Estatísticas de tarifa por sobrevivência
fare_stats = df_clean.groupby('Survived')['Fare'].describe()
print("\nEstatísticas da tarifa por sobrevivência:")
print(fare_stats)

# Correlação usando numpy
fare_values = df_clean['Fare'].values
survived_values = df_clean['Survived'].values
correlation = np.corrcoef(fare_values, survived_values)[0, 1]
print(f"\nCorrelação entre tarifa e sobrevivência: {correlation:.4f}")

# Média de tarifa por classe e sobrevivência
fare_class_surv = df_clean.pivot_table(values='Fare', 
                                        index='Pclass', 
                                        columns='Survived', 
                                        aggfunc='median')

fare_class_surv.columns = ['Não Sobreviveu', 'Sobreviveu']
print("\nMediana da tarifa por classe e sobrevivência:")
print(fare_class_surv.round(2))

# Análise adicional: faixas de tarifa personalizadas
bins_fare = [0, 10, 30, 100, 600]
labels_fare = ['Baixa (0-10)', 'Média (10-30)', 'Alta (30-100)', 'Premium (100+)']
df_clean['Faixa_Tarifa'] = pd.cut(df_clean['Fare'], bins=bins_fare, labels=labels_fare, right=False)

taxa_faixa_tarifa = df_clean.groupby('Faixa_Tarifa', observed=False)['Survived'].agg(['mean', 'count'])
taxa_faixa_tarifa['mean'] = (taxa_faixa_tarifa['mean'] * 100).round(2)
print("\nTaxa de sobrevivência por faixa de tarifa:")
print(taxa_faixa_tarifa)

print("\n📊 Resposta:")
print("  Sim, existe uma relação significativa entre a tarifa paga e a sobrevivência.")
print(f"  - A correlação é de {correlation:.4f} (positiva moderada)")
print(f"  - Passageiros que sobreviveram pagaram em média {fare_stats.loc[1, 'mean']:.2f}")
print(f"  - Passageiros que não sobreviveram pagaram em média {fare_stats.loc[0, 'mean']:.2f}")
print("  - Em todas as classes, quem sobreviveu tende a ter pago tarifas mais altas")
print("  - Conclusão: Tarifas mais altas estão associadas a maiores chances de sobrevivência,")
print("    provavelmente por estarem ligadas a cabines em locais mais favoráveis à evacuação")

# Visualização em texto
print("\n  Gráfico Comparativo de Tarifas (Texto):")
for survived in [0, 1]:
    label = 'Sobreviveu' if survived == 1 else 'Não Sobreviveu'
    mean_fare = fare_stats.loc[survived, 'mean']
    bar_length = int((mean_fare / fare_stats['mean'].max()) * 50)
    bar = '█' * bar_length
    print(f"  {label:15} | {bar} ${mean_fare:.2f}")

# ============================================================
# PERGUNTA 5: Qual o impacto do tamanho da família na sobrevivência?
# ============================================================
print("\n" + "-" * 80)
print("PERGUNTA 5: Qual o impacto do tamanho da família na sobrevivência?")
print("-" * 80)

# Análise detalhada
family_analysis = df_clean.groupby('Tamanho_Familia').agg({
    'Survived': ['mean', 'count', 'sum'],
    'PassengerId': 'count'
}).round(4)

family_analysis.columns = ['Taxa_Sobrevivencia', 'Total', 'Sobreviventes', 'Total_2']
family_analysis = family_analysis.drop(columns=['Total_2'])
family_analysis['Taxa_Sobrevivencia'] = (family_analysis['Taxa_Sobrevivencia'] * 100).round(2)
family_analysis['Percentual_Total'] = ((family_analysis['Total'] / len(df_clean)) * 100).round(2)

print("\nAnálise de sobrevivência por tamanho da família:")
print(family_analysis)

# Comparação Sozinho vs Acompanhado
sozinho = df_clean[df_clean['Categoria_Familia'] == 'Sozinho']['Survived'].mean() * 100
acompanhado = df_clean[df_clean['Categoria_Familia'] != 'Sozinho']['Survived'].mean() * 100

# Análise combinada: família e classe
familia_classe = df_clean.pivot_table(values='Survived', 
                                       index='Categoria_Familia', 
                                       columns='Pclass', 
                                       aggfunc='mean') * 100
familia_classe.columns = ['1ª Classe', '2ª Classe', '3ª Classe']

print("\nTaxa de sobrevivência por categoria familiar e classe (%):")
print(familia_classe.round(2))

# Análise de pais com filhos vs sem filhos
com_filhos = df_clean[df_clean['Parch'] > 0]['Survived'].mean() * 100
sem_filhos = df_clean[df_clean['Parch'] == 0]['Survived'].mean() * 100

print(f"\n  - Passageiros com filhos: Taxa de sobrevivência de {com_filhos:.2f}%")
print(f"  - Passageiros sem filhos: Taxa de sobrevivência de {sem_filhos:.2f}%")

print("\n📊 Resposta:")
print(f"  - Passageiros sozinhos: Taxa de sobrevivência de {sozinho:.2f}%")
print(f"  - Passageiros acompanhados: Taxa de sobrevivência de {acompanhado:.2f}%")
print(f"  - Melhor tamanho de família: {family_analysis['Taxa_Sobrevivencia'].idxmax()} membros")
print(f"    (Taxa: {family_analysis['Taxa_Sobrevivencia'].max():.2f}%)")
print(f"  - Pior tamanho de família: {family_analysis['Taxa_Sobrevivencia'].idxmin()} membros")
print(f"    (Taxa: {family_analysis['Taxa_Sobrevivencia'].min():.2f}%)")
print("\n  Conclusão: Famílias com 2-3 membros tiveram as maiores taxas de sobrevivência.")
print("  Famílias muito grandes (acima de 4 membros) tiveram dificuldade em se manter")
print("  unidas durante a evacuação, resultando em menor sobrevivência.")
print("  Passageiros sozinhos também tiveram taxas mais baixas, possivelmente por")
print("  falta de rede de apoio durante o caos do naufrágio.")

# Visualização em texto
print("\n  Gráfico de Barras - Tamanho da Família (Texto):")
max_rate = family_analysis['Taxa_Sobrevivencia'].max()
for tamanho in sorted(family_analysis.index):
    rate = family_analysis.loc[tamanho, 'Taxa_Sobrevivencia']
    total = int(family_analysis.loc[tamanho, 'Total'])
    bar_length = int((rate / max_rate) * 50)
    bar = '█' * bar_length
    print(f"  {tamanho} membro(s) | {bar} {rate:.1f}% (n={total})")

# ============================================================================
# 5. ANÁLISES ADICIONAIS COM PANDAS E NUMPY
# ============================================================================

print("\n" + "=" * 80)
print("5. ANÁLISES ADICIONAIS")
print("=" * 80)

# 5.1. Distribuição de idades por classe
print("\n5.1. Estatísticas de idade por classe:")
age_stats = df_clean.groupby('Pclass')['Age'].describe()
print(age_stats)

# 5.2. Passageiros com familiares vs sozinhos
print("\n5.2. Análise de acompanhantes:")
df_clean['Tem_Acompanhante'] = (df_clean['SibSp'] + df_clean['Parch']) > 0
acompanhante_stats = df_clean.groupby('Tem_Acompanhante')['Survived'].agg(['mean', 'count', 'sum'])
acompanhante_stats['mean'] = (acompanhante_stats['mean'] * 100).round(2)
acompanhante_stats.index = ['Sozinho', 'Acompanhado']
print(acompanhante_stats)

# 5.3. Análise de sobrevivência por porto de embarque e sexo
print("\n5.3. Sobrevivência por porto e sexo:")
porto_sexo = df_clean.pivot_table(values='Survived', 
                                   index='Embarked', 
                                   columns='Sex', 
                                   aggfunc='mean') * 100
print(porto_sexo.round(2))

# 5.4. Top 10 tarifas mais altas e sua sobrevivência
print("\n5.4. Top 10 passageiros com tarifas mais altas:")
top_fares = df_clean.nlargest(10, 'Fare')[['Name', 'Pclass', 'Fare', 'Survived', 'Sex', 'Age']]
top_fares['Survived'] = top_fares['Survived'].map({0: 'Não', 1: 'Sim'})
print(top_fares.to_string())

# 5.5. Análise de crianças (menores de 12 anos)
print("\n5.5. Análise de crianças (menores de 12 anos):")
criancas = df_clean[df_clean['Age'] < 12]
criancas_stats = criancas['Survived'].agg(['mean', 'count', 'sum'])
criancas_stats['mean'] = criancas_stats['mean'] * 100
print(f"  Total de crianças: {criancas_stats['count']}")
print(f"  Sobreviventes: {criancas_stats['sum']} ({criancas_stats['mean']:.2f}%)")

# Sobrevivência de crianças por classe
criancas_classe = criancas.groupby('Pclass')['Survived'].mean() * 100
print("\n  Sobrevivência de crianças por classe:")
for classe in [1, 2, 3]:
    if classe in criancas_classe.index:
        print(f"  - {classe}ª Classe: {criancas_classe[classe]:.2f}%")

# 5.6. Teste de hipótese simples (diferença de médias)
print("\n5.6. Análise estatística com NumPy:")
# Diferença de tarifa média entre sobreviventes e não sobreviventes
fare_survived = df_clean[df_clean['Survived'] == 1]['Fare'].values
fare_not_survived = df_clean[df_clean['Survived'] == 0]['Fare'].values

diff_means = np.mean(fare_survived) - np.mean(fare_not_survived)
# Cálculo do desvio padrão da diferença (aproximação)
se = np.sqrt(np.var(fare_survived)/len(fare_survived) + np.var(fare_not_survived)/len(fare_not_survived))
t_stat = diff_means / se

print(f"  Diferença de tarifa média (Sobreviventes - Não Sobreviventes): ${diff_means:.2f}")
print(f"  Estatística t aproximada: {t_stat:.2f}")
print(f"  Interpretação: A diferença é {'significativa' if abs(t_stat) > 2 else 'não significativa'}")

# ============================================================================
# 6. RESUMO FINAL E INSIGHTS
# ============================================================================

print("\n" + "=" * 80)
print("6. RESUMO FINAL E INSIGHTS ADICIONAIS")
print("=" * 80)

print("\n6.1. Principais Fatores de Sobrevivência (ordenados por importância):")
print("  1. Sexo: Mulheres tiveram prioridade no resgate")
print("  2. Classe Social: Passageiros da 1ª classe tiveram mais acesso aos botes")
print("  3. Tarifa: Correlacionada com a classe, indicou melhor localização no navio")
print("  4. Tamanho da Família: Famílias pequenas (2-3) tiveram melhor coordenação")
print("  5. Idade: Crianças tiveram maior chance de sobrevivência")
print("  6. Porto de Embarque: Passageiros de Cherbourg (C) tiveram maior sobrevivência")

print("\n6.2. Estatísticas Gerais do Dataset:")
print(f"  - Total de passageiros: {len(df_clean)}")
print(f"  - Sobreviventes: {df_clean['Survived'].sum()} ({df_clean['Survived'].mean()*100:.2f}%)")
print(f"  - Não sobreviventes: {len(df_clean) - df_clean['Survived'].sum()} ({(1-df_clean['Survived'].mean())*100:.2f}%)")
print(f"  - Idade média: {df_clean['Age'].mean():.1f} anos")
print(f"  - Idade mediana: {df_clean['Age'].median():.1f} anos")
print(f"  - Tarifa média: ${df_clean['Fare'].mean():.2f}")
print(f"  - Tarifa mediana: ${df_clean['Fare'].median():.2f}")
print(f"  - Distribuição por classe: 1ª={len(df_clean[df_clean['Pclass']==1])} | "
      f"2ª={len(df_clean[df_clean['Pclass']==2])} | "
      f"3ª={len(df_clean[df_clean['Pclass']==3])}")
print(f"  - Distribuição por sexo: Masculino={len(df_clean[df_clean['Sex']=='male'])} | "
      f"Feminino={len(df_clean[df_clean['Sex']=='female'])}")

print("\n6.3. Features Engineering (Novas colunas criadas):")
print("  - Faixa_Etaria: Categorização da idade")
print("  - Categoria_Tarifa: Quartis da tarifa")
print("  - Tamanho_Familia: SibSp + Parch + 1")
print("  - Categoria_Familia: Sozinho/Família Pequena/Família Grande")
print("  - Titulo: Extraído do nome do passageiro")
print("  - Faixa_Tarifa: Faixas personalizadas de tarifa")
print("  - Tem_Acompanhante: Indicador de acompanhante")

print("\n6.4. Recomendações para Modelagem:")
print("  - Para modelos de ML: Feature engineering com estas variáveis")
print("  - Considerar interações: Sexo*Classe, Idade*Sexo, Classe*Tarifa")
print("  - Tratar outliers na tarifa (possíveis erros de registro)")
print("  - Codificar variáveis categóricas: Sex, Embarked, Faixa_Etaria, Titulo")
print("  - Feature importance provavelmente seguirá: Sex > Pclass > Fare > Age")

# ============================================================================
# 7. EXPORTAÇÃO
# ============================================================================

print("\n" + "=" * 80)
print("7. EXPORTAÇÃO DOS DADOS")
print("=" * 80)

# Exportar dataset limpo
df_clean.to_csv('Titanic_Limpo.csv', index=False)
print("\n✓ Dataset limpo exportado como 'Titanic_Limpo.csv'")

# Exportar resumo das análises
with open('Resumo_Analise_Titanic.txt', 'w', encoding='utf-8') as f:
    f.write("RESUMO DA ANÁLISE DO TITANIC\n")
    f.write("=" * 80 + "\n\n")
    
    f.write("1. SOBREVIVÊNCIA POR CLASSE\n")
    f.write(taxa_sobrevivencia_classe.to_string())
    f.write("\n\n")
    
    f.write("2. SOBREVIVÊNCIA POR SEXO\n")
    f.write(taxa_sobrevivencia_sexo.to_string())
    f.write("\n\n")
    
    f.write("3. SOBREVIVÊNCIA POR FAIXA ETÁRIA\n")
    f.write(taxa_faixa_etaria.to_string())
    f.write("\n\n")
    
    f.write("4. SOBREVIVÊNCIA POR TAMANHO DA FAMÍLIA\n")
    f.write(family_analysis.to_string())
    f.write("\n\n")
    
    f.write("5. CORRELAÇÃO TARIFA-SOBREVIVÊNCIA\n")
    f.write(f"Correlação: {correlation:.4f}\n")
    f.write(fare_stats.to_string())

print("✓ Resumo das análises exportado como 'Resumo_Analise_Titanic.txt'")

print("\n" + "=" * 80)
print("ANÁLISE CONCLUÍDA COM SUCESSO!")
print("=" * 80)
print(f"\nArquivos gerados:")
print("  -")

ANÁLISE EXPLORATÓRIA E TRATAMENTO DE DADOS DO TITANIC

2. DESCRIÇÃO DOS DADOS

2.1. Primeiras linhas do dataset:
   PassengerId  Survived  Pclass                                          Name     Sex   Age  SibSp  Parch     Ticket  Fare Cabin Embarked
0          892         0       3                              Kelly, Mr. James    male 34.50      0      0     330911  7.83   NaN        Q
1          893         1       3              Wilkes, Mrs. James (Ellen Needs)  female 47.00      1      0     363272  7.00   NaN        S
2          894         0       2                     Myles, Mr. Thomas Francis    male 62.00      0      0     240276  9.69   NaN        Q
3          895         0       3                              Wirz, Mr. Albert    male 27.00      0      0     315154  8.66   NaN        S
4          896         1       3  Hirvonen, Mrs. Alexander (Helga E Lindqvist)  female 22.00      1      1    3101298 12.29   NaN        S
5          897         0       3                    S

C:\Users\ralve\AppData\Local\Temp\ipykernel_1056\186915378.py:67: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns


             mean  count  sum
Sozinho     26.88    253   68
Acompanhado 50.91    165   84

5.3. Sobrevivência por porto e sexo:
Sex       female  male
Embarked              
C         100.00  0.00
Q         100.00  0.00
S         100.00  0.00

5.4. Top 10 passageiros com tarifas mais altas:
                                                                Name  Pclass   Fare Survived     Sex   Age
343  Cardeza, Mrs. James Warburton Martinez (Charlotte Wardle Drake)       1 512.33      Sim  female 58.00
53                                        Fortune, Miss. Ethel Flora       1 263.00      Sim  female 28.00
69                               Fortune, Mrs. Mark (Mary McDougald)       1 263.00      Sim  female 60.00
24                   Ryerson, Mrs. Arthur Larned (Emily Maria Borie)       1 262.38      Sim  female 48.00
59                                       Chaudanson, Miss. Victorine       1 262.38      Sim  female 36.00
64                                       Ryerson, Master. John Bor